# Test Enrichissement Annuaire Avocats
Test sur 10 lignes : récupérer la taille des cabinets (API Entreprises) et l'URL du site internet (Perplexity Sonar Pro)

In [1]:
import csv
import json
import time
import requests
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()

PERPLEXITY_API_KEY = os.getenv("PERPLEXITY_API_KEY")
print(f"Perplexity API key loaded: {'Yes' if PERPLEXITY_API_KEY and PERPLEXITY_API_KEY != 'your_perplexity_api_key_here' else 'No - update .env!'}")

Perplexity API key loaded: Yes


## 1. Lecture des 10 premières lignes du CSV

In [2]:
CSV_PATH = "annuaire-avocats-20260316.csv"

cabinets = []
with open(CSV_PATH, encoding="utf-8-sig") as f:
    reader = csv.DictReader(f, delimiter=";")
    for i, row in enumerate(reader):
        if i >= 10:
            break
        cabinets.append({
            "raison_sociale": row["Raison Sociale"].strip(),
            "siret_siren": row["Siret Siren"].strip(),
            "adresse": f"{row['Adresse 1'].strip()} {row['Adresse 2'].strip()}".strip(),
            "cp": row["Cp"].strip(),
            "ville": row["Ville"].strip(),
            "site_csv": row["Site internet du cabinet "].strip() if row.get("Site internet du cabinet ") else ""
        })

print(f"{len(cabinets)} cabinets extraits:\n")
for i, c in enumerate(cabinets):
    print(f"{i+1}. {c['raison_sociale']} | SIREN: {c['siret_siren']} | {c['ville']} | Site CSV: {c['site_csv'] or 'N/A'}")

10 cabinets extraits:

1. AZZOUZ DAHAB | SIREN: 988689832 | AGEN | Site CSV: N/A
2. BADY AURÉLIA | SIREN: 844832543 | AGEN | Site CSV: N/A
3. VALAY-BELACEL-DELBREL-CERDAN | SIREN: 488896952 | MARMANDE | Site CSV: N/A
4. BELLANDI | SIREN: 424260057 | AGEN | Site CSV: N/A
5. BERT AVOCAT | SIREN: 940166036 | AGEN | Site CSV: N/A
6. BICKART-MAGNES | SIREN: 449534262 | BON ENCONTRE | Site CSV: N/A
7. JURI-LAWYERS CONSULTANTS | SIREN: 387565856 | MARMANDE | Site CSV: N/A
8. SELARL LEX ALLIANCE | SIREN: 500184726 | AGEN | Site CSV: N/A
9. BOSSUT PHILIPPE | SIREN: 381129949 | AYET | Site CSV: N/A
10. BOUCHINDHOMME AGATHE | SIREN: 922521406 | AGEN | Site CSV: N/A


## 2. Fonction API Recherche Entreprises (gouv.fr)

In [3]:
API_ENTREPRISES_URL = "https://recherche-entreprises.api.gouv.fr/search"

TRANCHES_EFFECTIFS = {
    "00": "0 salarié",
    "01": "1-2 salariés",
    "02": "3-5 salariés",
    "03": "6-9 salariés",
    "11": "10-19 salariés",
    "12": "20-49 salariés",
    "21": "50-99 salariés",
    "22": "100-199 salariés",
    "31": "200-249 salariés",
    "32": "250-499 salariés",
    "41": "500-999 salariés",
    "42": "1000-1999 salariés",
    "51": "2000-4999 salariés",
    "52": "5000-9999 salariés",
    "53": "10000+ salariés",
}


def get_info_entreprise(siren: str) -> dict | None:
    """Récupère les infos d'une entreprise via l'API Recherche Entreprises."""
    try:
        resp = requests.get(API_ENTREPRISES_URL, params={"q": siren}, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        
        if not data.get("results"):
            return None
        
        entreprise = data["results"][0]
        siege = entreprise.get("siege", {})
        
        # Dirigeants
        dirigeants = []
        for d in entreprise.get("dirigeants", []):
            if d.get("type_dirigeant") == "personne physique":
                dirigeants.append(f"{d.get('prenoms', '')} {d.get('nom', '')}")
            else:
                dirigeants.append(d.get("denomination", "N/A"))
        
        tranche = entreprise.get("tranche_effectif_salarie") or siege.get("tranche_effectif_salarie", "")
        
        return {
            "siren": entreprise.get("siren", ""),
            "nom_complet": entreprise.get("nom_complet", ""),
            "activite_principale": entreprise.get("activite_principale", ""),
            "nature_juridique": entreprise.get("nature_juridique", ""),
            "nb_etablissements": entreprise.get("nombre_etablissements", 0),
            "tranche_effectifs": TRANCHES_EFFECTIFS.get(str(tranche), f"Code: {tranche}"),
            "dirigeants": dirigeants,
            "adresse": siege.get("adresse", ""),
            "date_creation": entreprise.get("date_creation", ""),
            "etat": entreprise.get("etat_administratif", ""),
        }
    except Exception as e:
        print(f"  Erreur API Entreprises pour {siren}: {e}")
        return None


# Test rapide
test = get_info_entreprise("988689832")
print(json.dumps(test, indent=2, ensure_ascii=False))

{
  "siren": "988689832",
  "nom_complet": "DAHAB AZZOUZ",
  "activite_principale": "69.10Z",
  "nature_juridique": "1000",
  "nb_etablissements": 1,
  "tranche_effectifs": "Code: NN",
  "dirigeants": [
    "DAHAB AZZOUZ"
  ],
  "adresse": "28 RUE MONTESQUIEU 47000 AGEN",
  "date_creation": "2025-07-01",
  "etat": "A"
}


## 3. Test API Entreprises sur les 10 cabinets

In [4]:
resultats_api = []

for i, cab in enumerate(cabinets):
    print(f"[{i+1}/10] {cab['raison_sociale']} (SIREN: {cab['siret_siren']})...")
    info = get_info_entreprise(cab["siret_siren"])
    resultats_api.append(info)
    
    if info:
        print(f"  -> {info['nom_complet']} | {info['tranche_effectifs']} | {info['nb_etablissements']} établissement(s) | Activité: {info['activite_principale']}")
        print(f"     Dirigeants: {', '.join(info['dirigeants']) if info['dirigeants'] else 'N/A'}")
        print(f"     État: {info['etat']} | Créé le: {info['date_creation']}")
    else:
        print(f"  -> Aucun résultat trouvé")
    
    time.sleep(0.15)  # Rate limiting

print(f"\n--- Résumé: {sum(1 for r in resultats_api if r)}/10 entreprises trouvées ---")

[1/10] AZZOUZ DAHAB (SIREN: 988689832)...
  -> DAHAB AZZOUZ | Code: NN | 1 établissement(s) | Activité: 69.10Z
     Dirigeants: DAHAB AZZOUZ
     État: A | Créé le: 2025-07-01
[2/10] BADY AURÉLIA (SIREN: 844832543)...
  -> AURELIA BADY | 1-2 salariés | 3 établissement(s) | Activité: 69.10Z
     Dirigeants: AURELIA ELODIE BADY
     État: A | Créé le: 2019-01-02
[3/10] VALAY-BELACEL-DELBREL-CERDAN (SIREN: 488896952)...
  -> VALAY BELACEL DELBREL CERDAN (VBDC) | 3-5 salariés | 4 établissement(s) | Activité: 69.10Z
     Dirigeants: YANN DELBREL, VIRGINIE KATHIA BELACEL (GALISSAIRE), LUDOVIC PIERRE ANDRE VALAY
     État: A | Créé le: 2006-01-01
[4/10] BELLANDI (SIREN: 424260057)...
  -> PHILIPPE BELLANDI | Code: NN | 3 établissement(s) | Activité: 69.10Z
     Dirigeants: PHILIPPE STEPHANE BELLANDI
     État: A | Créé le: 1999-01-25
[5/10] BERT AVOCAT (SIREN: 940166036)...
  -> BERT AVOCAT | Code: NN | 1 établissement(s) | Activité: 69.10Z
     Dirigeants: GUILLAUME BERT
     État: A | Créé 

## 4. Fonction Perplexity Sonar Pro (recherche URL)

In [5]:
perplexity_client = OpenAI(
    api_key=PERPLEXITY_API_KEY,
    base_url="https://api.perplexity.ai"
)


def find_website(raison_sociale: str, ville: str, siren: str) -> dict:
    """Recherche le site internet d'un cabinet d'avocats via Perplexity Sonar Pro."""
    prompt = (
        f"Trouve le site internet officiel du cabinet d'avocats '{raison_sociale}' "
        f"situé à {ville} (SIREN: {siren}). "
        f"Réponds UNIQUEMENT avec un JSON au format: "
        f'{{"url": "https://...", "confiance": "haute/moyenne/basse"}} '
        f"Si tu ne trouves pas de site, réponds: {{\"url\": null, \"confiance\": \"aucune\"}}"
    )
    
    try:
        response = perplexity_client.chat.completions.create(
            model="sonar-pro",
            messages=[
                {"role": "system", "content": "Tu es un assistant qui recherche les sites internet de cabinets d'avocats français. Réponds uniquement en JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,
        )
        
        content = response.choices[0].message.content.strip()
        # Extraire le JSON de la réponse (peut être entouré de ```json ... ```)
        if "```" in content:
            content = content.split("```json")[-1].split("```")[0].strip()
            if not content:
                content = response.choices[0].message.content.split("```")[1].strip()
        
        result = json.loads(content)
        return result
    except json.JSONDecodeError:
        # Si le JSON ne parse pas, essayer d'extraire l'URL manuellement
        import re
        urls = re.findall(r'https?://[^\s\"\}]+', content)
        return {"url": urls[0] if urls else None, "confiance": "basse", "raw": content}
    except Exception as e:
        return {"url": None, "confiance": "erreur", "erreur": str(e)}


# Test rapide sur le premier cabinet
test_url = find_website(cabinets[0]["raison_sociale"], cabinets[0]["ville"], cabinets[0]["siret_siren"])
print(f"Test: {cabinets[0]['raison_sociale']}")
print(json.dumps(test_url, indent=2, ensure_ascii=False))

Test: AZZOUZ DAHAB
{
  "url": null,
  "confiance": "aucune"
}


## 5. Test Perplexity sur les 10 cabinets

In [6]:
resultats_url = []

for i, cab in enumerate(cabinets):
    print(f"[{i+1}/10] Recherche URL pour {cab['raison_sociale']} ({cab['ville']})...")
    result = find_website(cab["raison_sociale"], cab["ville"], cab["siret_siren"])
    resultats_url.append(result)
    
    url = result.get("url", "N/A")
    confiance = result.get("confiance", "?")
    print(f"  -> URL: {url} (confiance: {confiance})")
    
    time.sleep(1)  # Rate limiting Perplexity

urls_trouvees = sum(1 for r in resultats_url if r.get("url"))
print(f"\n--- Résumé: {urls_trouvees}/10 URLs trouvées ---")

[1/10] Recherche URL pour AZZOUZ DAHAB (AGEN)...
  -> URL: None (confiance: aucune)
[2/10] Recherche URL pour BADY AURÉLIA (AGEN)...
  -> URL: https://bady-avocat-agen.fr (confiance: haute)
[3/10] Recherche URL pour VALAY-BELACEL-DELBREL-CERDAN (MARMANDE)...
  -> URL: https://www.vbd-avocats.fr (confiance: haute)
[4/10] Recherche URL pour BELLANDI (AGEN)...
  -> URL: None (confiance: aucune)
[5/10] Recherche URL pour BERT AVOCAT (AGEN)...
  -> URL: https://bertavocat.fr (confiance: haute)
[6/10] Recherche URL pour BICKART-MAGNES (BON ENCONTRE)...
  -> URL: https://avocat-bickart-magnes.fr (confiance: haute)
[7/10] Recherche URL pour JURI-LAWYERS CONSULTANTS (MARMANDE)...
  -> URL: https://www.cabinet-jlc.com (confiance: haute)
[8/10] Recherche URL pour SELARL LEX ALLIANCE (AGEN)...
  -> URL: None (confiance: aucune)
[9/10] Recherche URL pour BOSSUT PHILIPPE (AYET)...
  -> URL: None (confiance: aucune)
[10/10] Recherche URL pour BOUCHINDHOMME AGATHE (AGEN)...
  -> URL: None (confiance: 

## 6. Récapitulatif complet

In [7]:
print("=" * 120)
print("RÉCAPITULATIF ENRICHISSEMENT - 10 PREMIERS CABINETS")
print("=" * 120)

for i, cab in enumerate(cabinets):
    api = resultats_api[i] if i < len(resultats_api) else None
    url_info = resultats_url[i] if i < len(resultats_url) else {}
    
    print(f"\n{'─' * 80}")
    print(f"Cabinet {i+1}: {cab['raison_sociale']}")
    print(f"{'─' * 80}")
    print(f"  SIREN:      {cab['siret_siren']}")
    print(f"  Adresse:    {cab['adresse']}, {cab['cp']} {cab['ville']}")
    print(f"  Site (CSV): {cab['site_csv'] or 'Non renseigné'}")
    
    if api:
        print(f"  --- API Entreprises ---")
        print(f"  Nom complet:      {api['nom_complet']}")
        print(f"  Taille:           {api['tranche_effectifs']}")
        print(f"  Nb établissements:{api['nb_etablissements']}")
        print(f"  Activité NAF:     {api['activite_principale']}")
        print(f"  Dirigeants:       {', '.join(api['dirigeants']) if api['dirigeants'] else 'N/A'}")
        print(f"  État:             {api['etat']}")
        print(f"  Date création:    {api['date_creation']}")
    else:
        print(f"  --- API Entreprises: Aucune donnée ---")
    
    url = url_info.get("url", "N/A")
    confiance = url_info.get("confiance", "?")
    print(f"  --- Perplexity ---")
    print(f"  Site trouvé:  {url}")
    print(f"  Confiance:    {confiance}")

print(f"\n{'=' * 120}")
print(f"STATS FINALES:")
print(f"  Entreprises trouvées (API):  {sum(1 for r in resultats_api if r)}/10")
print(f"  URLs trouvées (Perplexity):  {sum(1 for r in resultats_url if r.get('url'))}/10")
print(f"{'=' * 120}")

RÉCAPITULATIF ENRICHISSEMENT - 10 PREMIERS CABINETS

────────────────────────────────────────────────────────────────────────────────
Cabinet 1: AZZOUZ DAHAB
────────────────────────────────────────────────────────────────────────────────
  SIREN:      988689832
  Adresse:    28 rue Montesquieu, 47000 AGEN
  Site (CSV): Non renseigné
  --- API Entreprises ---
  Nom complet:      DAHAB AZZOUZ
  Taille:           Code: NN
  Nb établissements:1
  Activité NAF:     69.10Z
  Dirigeants:       DAHAB AZZOUZ
  État:             A
  Date création:    2025-07-01
  --- Perplexity ---
  Site trouvé:  None
  Confiance:    aucune

────────────────────────────────────────────────────────────────────────────────
Cabinet 2: BADY AURÉLIA
────────────────────────────────────────────────────────────────────────────────
  SIREN:      844832543
  Adresse:    58 rue Mirabeau, 47000 AGEN
  Site (CSV): Non renseigné
  --- API Entreprises ---
  Nom complet:      AURELIA BADY
  Taille:           1-2 salariés
  N

## 7. Agent Crawl4AI + LangChain — Extraction Spécialités & Équipe

L'agent reçoit une URL, crawle la page d'accueil, analyse le markdown, et décide s'il doit explorer d'autres pages pour trouver les **spécialités** et l'**équipe** du cabinet. Max **5 pages** par site.

In [8]:
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from markdownify import markdownify as md
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
print(f"Anthropic API key loaded: {'Yes' if ANTHROPIC_API_KEY else 'No - update .env!'}")

# --- LLM ---
llm = ChatAnthropic(
    model="claude-sonnet-4-20250514",
    api_key=ANTHROPIC_API_KEY,
    max_tokens=4096,
)

# --- Outil Crawl (requests + markdownify, pas besoin de Playwright) ---
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"
}

@tool
def crawl_page(url: str) -> str:
    """Crawle une page web et retourne son contenu en markdown.
    Utilise cet outil pour explorer les pages d'un site de cabinet d'avocats.
    Tu peux crawler jusqu'a 5 pages par site (la page d'accueil + 4 sous-pages).
    """
    print(f"  [CRAWL] {url}")
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15, allow_redirects=True)
        resp.raise_for_status()
        resp.encoding = resp.apparent_encoding

        soup = BeautifulSoup(resp.text, "html.parser")
        # Supprimer les elements non pertinents
        for tag in soup(["script", "style", "nav", "footer", "iframe", "noscript"]):
            tag.decompose()

        markdown = md(str(soup.body or soup), strip=["img"])
        # Tronquer a 15000 chars pour ne pas exploser le contexte
        if len(markdown) > 15000:
            markdown = markdown[:15000] + "\n\n[... contenu tronque ...]"
        return markdown
    except requests.exceptions.Timeout:
        return f"Erreur: Timeout lors du chargement de {url}"
    except requests.exceptions.HTTPError as e:
        return f"Erreur HTTP {e.response.status_code} pour {url}"
    except Exception as e:
        return f"Erreur crawl: {type(e).__name__}: {e}"

# Test rapide
test_crawl = crawl_page.invoke("https://bady-avocat-agen.fr")
print(f"Crawl OK - {len(test_crawl)} caracteres")
print(test_crawl[:500] + "\n...")

c:\Users\Jordi\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Anthropic API key loaded: Yes
  [CRAWL] https://bady-avocat-agen.fr
Crawl OK - 3388 caracteres
[Skip to content](#main)

[Menu
Fermer](#)

AURÉLIA BADY

Avocat à Agen
-------------

Avocat au barreau d’AGEN, Maître Aurélia BADY conseille et représente devant les Juridictions les intérêts des particuliers et des professionnels.

Le Cabinet intervient principalement dans les domaines du droit de la famille, du divorce et de la séparation.

[En savoir plus](#cabinet)

[Prenez Contact](https://bady-avocat-agen.fr/contact/)

Le Cabinet
----------

Titulaire d’un Master 2 en droit 
...


In [9]:
AGENT_SYSTEM_PROMPT = """Tu es un agent specialise dans l'extraction d'informations depuis les sites web de cabinets d'avocats francais.

## Ton objectif
Extraire deux types d'informations :
1. **SPECIALITES** : les domaines de droit pratiques (droit penal, droit de la famille, droit des affaires, droit du travail, etc.)
2. **EQUIPE** : la liste des avocats/collaborateurs avec leurs noms et roles si disponibles

## Ta methode
1. Commence TOUJOURS par crawler la page d'accueil avec l'outil crawl_page
2. Analyse le markdown retourne :
   - Si tu trouves les specialites ET l'equipe -> extrais-les directement
   - Si tu ne trouves pas tout -> cherche dans le markdown les liens internes pertinents
     (ex: /specialites, /domaines-de-competences, /equipe, /avocats, /notre-equipe, /a-propos, /nos-competences, /expertises)
   - Crawle ces pages pour completer tes informations
3. Tu as un MAXIMUM de 5 appels a crawl_page au total (page d'accueil incluse)
4. Les URLs relatives doivent etre construites a partir du domaine de base

## Format de reponse finale
Quand tu as fini, reponds avec EXACTEMENT ce format JSON (et rien d'autre) :
```json
{
    "specialites": ["droit penal", "droit de la famille"],
    "equipe": [{"nom": "Jean Dupont", "role": "Associe"}],
    "pages_crawlees": 3,
    "notes": "informations complementaires eventuelles"
}
```

Si le site est inaccessible ou ne contient aucune info utile :
```json
{
    "specialites": [],
    "equipe": [],
    "pages_crawlees": 1,
    "notes": "Site inaccessible ou aucune information trouvee"
}
```"""

agent = create_react_agent(
    llm,
    tools=[crawl_page],
    prompt=AGENT_SYSTEM_PROMPT,
)

print("Agent LangChain cree (Claude Sonnet + crawl_page tool, max 5 pages/site)")

Agent LangChain cree (Claude Sonnet + crawl_page tool, max 5 pages/site)


C:\Users\Jordi\AppData\Local\Temp\ipykernel_20452\3298593784.py:39: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [10]:
def extract_cabinet_info(url: str, raison_sociale: str) -> dict:
    """Lance l'agent sur un site de cabinet pour extraire specialites + equipe."""
    print(f"\n{'='*60}")
    print(f"AGENT -> {raison_sociale} ({url})")
    print(f"{'='*60}")

    try:
        result = agent.invoke(
            {"messages": [("user", f"Analyse le site du cabinet d'avocats '{raison_sociale}' a cette URL : {url}\nExtrais les specialites et l'equipe.")]},
            config={"recursion_limit": 15}
        )

        # Recuperer le dernier message de l'agent
        last_msg = result["messages"][-1].content
        print(f"\n  Reponse agent (extrait) : {last_msg[:200]}...")

        # Parser le JSON de la reponse
        if "```json" in last_msg:
            json_str = last_msg.split("```json")[-1].split("```")[0].strip()
        elif "```" in last_msg:
            json_str = last_msg.split("```")[1].split("```")[0].strip()
        elif "{" in last_msg:
            start = last_msg.index("{")
            end = last_msg.rindex("}") + 1
            json_str = last_msg[start:end]
        else:
            json_str = last_msg

        parsed = json.loads(json_str)
        print(f"  Specialites trouvees : {len(parsed.get('specialites', []))}")
        print(f"  Membres equipe :      {len(parsed.get('equipe', []))}")
        print(f"  Pages crawlees :      {parsed.get('pages_crawlees', '?')}")
        return parsed

    except json.JSONDecodeError:
        print(f"  ERREUR: Impossible de parser le JSON de l'agent")
        return {"specialites": [], "equipe": [], "pages_crawlees": 0, "notes": f"Erreur parsing JSON. Reponse brute: {last_msg[:300]}"}
    except Exception as e:
        print(f"  ERREUR: {e}")
        return {"specialites": [], "equipe": [], "pages_crawlees": 0, "notes": str(e)}

# Test sur un seul cabinet (le premier avec une URL)
urls_disponibles = [(i, cab, resultats_url[i]) for i, cab in enumerate(cabinets) if resultats_url[i].get("url")]
if urls_disponibles:
    idx, cab_test, url_test = urls_disponibles[0]
    result_test = extract_cabinet_info(url_test["url"], cab_test["raison_sociale"])
    print(f"\n--- Resultat test ---")
    print(json.dumps(result_test, indent=2, ensure_ascii=False))
else:
    print("Aucune URL disponible pour tester l'agent")


AGENT -> BADY AURÉLIA (https://bady-avocat-agen.fr)
  [CRAWL] https://bady-avocat-agen.fr
  [CRAWL] https://bady-avocat-agen.fr/droits-du-divorce/
  [CRAWL] https://bady-avocat-agen.fr/droits-de-la-famille/
  [CRAWL] https://bady-avocat-agen.fr/droits-de-la-separation/

  Reponse agent (extrait) : Parfait ! J'ai maintenant toutes les informations nécessaires. D'après les 4 pages crawlées, j'ai une vision complète des spécialités du cabinet et de l'équipe. Le cabinet semble être un cabinet indiv...
  Specialites trouvees : 11
  Membres equipe :      1
  Pages crawlees :      4

--- Resultat test ---
{
  "specialites": [
    "droit de la famille",
    "droit du divorce",
    "droit de la séparation",
    "droit des successions",
    "droit des majeurs protégés",
    "droit de l'adoption",
    "droit des grands-parents",
    "assistance éducative",
    "PACS",
    "concubinage",
    "droit de la liquidation du régime matrimonial"
  ],
  "equipe": [
    {
      "nom": "Aurélia BADY",
    

In [11]:
# Lancer l'agent sur TOUS les cabinets ayant une URL
resultats_crawl = {}

for i, cab in enumerate(cabinets):
    url_info = resultats_url[i] if i < len(resultats_url) else {}
    url = url_info.get("url")

    if not url:
        print(f"[{i+1}/10] {cab['raison_sociale']} - Pas d'URL, skip")
        resultats_crawl[i] = {"specialites": [], "equipe": [], "pages_crawlees": 0, "notes": "Pas d'URL trouvee"}
        continue

    resultats_crawl[i] = extract_cabinet_info(url, cab["raison_sociale"])
    time.sleep(2)  # Pause entre chaque cabinet

# --- RECAP FINAL ---
print(f"\n\n{'='*100}")
print("RECAP FINAL : SPECIALITES & EQUIPE")
print(f"{'='*100}")

for i, cab in enumerate(cabinets):
    api = resultats_api[i] if i < len(resultats_api) else None
    url_info = resultats_url[i] if i < len(resultats_url) else {}
    crawl = resultats_crawl.get(i, {})

    print(f"\n{'─'*80}")
    print(f"Cabinet {i+1}: {cab['raison_sociale']} ({cab['ville']})")
    print(f"  SIREN: {cab['siret_siren']} | Taille: {api['tranche_effectifs'] if api else 'N/A'}")
    print(f"  URL:   {url_info.get('url', 'N/A')}")

    specs = crawl.get("specialites", [])
    equipe = crawl.get("equipe", [])
    print(f"  Specialites ({len(specs)}): {', '.join(specs) if specs else 'Aucune trouvee'}")
    print(f"  Equipe ({len(equipe)}):")
    for m in equipe:
        if isinstance(m, dict):
            print(f"    - {m.get('nom', '?')} ({m.get('role', 'N/A')})")
        else:
            print(f"    - {m}")
    if crawl.get("notes"):
        print(f"  Notes: {crawl['notes']}")

print(f"\n{'='*100}")
total_specs = sum(1 for r in resultats_crawl.values() if r.get("specialites"))
total_equipe = sum(1 for r in resultats_crawl.values() if r.get("equipe"))
print(f"Cabinets avec specialites: {total_specs}/10")
print(f"Cabinets avec equipe:      {total_equipe}/10")

[1/10] AZZOUZ DAHAB - Pas d'URL, skip

AGENT -> BADY AURÉLIA (https://bady-avocat-agen.fr)
  [CRAWL] https://bady-avocat-agen.fr
  [CRAWL] https://bady-avocat-agen.fr/droits-du-divorce/
  [CRAWL] https://bady-avocat-agen.fr/droits-de-la-famille/
  [CRAWL] https://bady-avocat-agen.fr/droits-de-la-separation/
  [CRAWL] https://bady-avocat-agen.fr/contact/

  Reponse agent (extrait) : Parfait ! J'ai maintenant toutes les informations nécessaires après avoir crawlé 5 pages du site. Je peux maintenant fournir une analyse complète du cabinet BADY AURÉLIA.

```json
{
    "specialites":...
  Specialites trouvees : 10
  Membres equipe :      1
  Pages crawlees :      5

AGENT -> VALAY-BELACEL-DELBREL-CERDAN (https://www.vbd-avocats.fr)
  [CRAWL] https://www.vbd-avocats.fr
  [CRAWL] https://www.vbd-avocats.fr/expertises.htm
  [CRAWL] https://www.vbd-avocats.fr/cabinet.htm
  [CRAWL] https://www.vbd-avocats.fr/page/annuaire/maitre-ludovic-valay-3.htm

  Reponse agent (extrait) : Excellent ! J'ai m